# MobileNetV3-Small Slot Occupancy & Fusion: Large-Scale Training Pipeline

This notebook runs the formal training pipeline on Google Colab (GPU) on a large sample from the official PKLot archive (~4.56 GB), sampled evenly across all 3 lots (`PUCPR`, `UFPR04`, `UFPR05`), weather conditions (`Sunny`, `Cloudy`, `Rainy`), and days.

### Safeguards & Key Features:
- **Reproducibility & Traceability**: Clones latest `main`, checks out clean code, and prints the exact git commit hash.
- **Cache Invalidation**: Deletes any stale cached crops (`data/cache/crops`) prior to training/evaluation.
- **Unified Preprocessing**: Enforces the shared `preprocess_slot_crop` everywhere.
- **Protocols**:
  1. **3-Fold Leave-One-Lot-Out (LOO)**: Evaluates generalization across lots with test sets on PUCPR, UFPR04, and UFPR05.
  2. **Final Model (`--protocol all`)**: Trains `mobilenetv3_final.pth` across all three lots for production deployment.
- **Verification Export**: Saves `repro_check.json` with 50 test predictions per fold to verify local reproducibility.


## 0. Training Configuration
Configure dataset sample size and hyperparameters here.


In [ ]:
# Visible top-level configuration
FRAMES_PER_LOT = 150       # Target sampled frames per lot (>=150 frames per lot across Sunny/Cloudy/Rainy/days)
EPOCHS = 8                 # Epochs per fold
BATCH_SIZE = 64            # Batch size for GPU training
LEARNING_RATE = 3e-4       # Learning rate for AdamW
SEED = 42                  # Global random seed

print(f"=== Pipeline Configuration ===")
print(f"Frames per lot:  {FRAMES_PER_LOT} (~{FRAMES_PER_LOT * 3} total frames)")
print(f"Epochs:          {EPOCHS}")
print(f"Batch size:      {BATCH_SIZE}")
print(f"Learning rate:   {LEARNING_RATE}")
print(f"Random seed:     {SEED}")


## 1. Environment Setup & GPU Verification


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available. Please change runtime type to GPU (T4).")


## 2. Clone Latest Repository & Print Git Commit Hash


In [ ]:
%cd /content
!if [ -d "smart-parking-platform" ]; then rm -rf smart-parking-platform; fi
!git clone https://github.com/bh00mi-11/smart-parking-platform.git
%cd /content/smart-parking-platform/vision

# Print current git commit hash for full auditability
import subprocess
commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("utf-8").strip()
branch_name = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode("utf-8").strip()
print(f"Running on branch: '{branch_name}' at commit: {commit_hash}")

!pip install -r requirements.txt


## 3. Cache Invalidation & Large Dataset Sampling
Deletes any stale crop caches, downloads the official PKLot archive (4.56 GB), and extracts a stratified sample evenly across weather and days with stride sampling.


In [ ]:
# Preprocessing Safeguard: Delete any cached crops to guarantee fresh preprocessing
!rm -rf /content/smart-parking-platform/vision/data/cache/crops
print("Cleaned /content/smart-parking-platform/vision/data/cache/crops")

# Download and sample official PKLot archive
!python scripts/download_data.py --official-archive --frames-per-lot {FRAMES_PER_LOT}

# Inspect dataset
!python scripts/inspect_data.py


## 4. 3-Fold Leave-One-Lot-Out (LOO) Training
Trains MobileNetV3-small slot classifiers on each fold, saves checkpoints, and exports 50 reference predictions to `repro_check.json`.


In [ ]:
for fold in [0, 1, 2]:
    print(f"\n{'='*70}\nSTARTING LEAVE-ONE-LOT-OUT (LOO) FOLD {fold}\n{'='*70}")
    !python src/train_classifier.py --epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LEARNING_RATE} --protocol loo --fold {fold} --seed {SEED} --export-repro --repro-samples 50


## 5. Train Final Production Model (`--protocol all`)
Trains `mobilenetv3_final.pth` across ALL three lots with validation partitioned strictly by day.


In [ ]:
print(f"\n{'='*70}\nSTARTING FINAL MODEL TRAINING (--protocol all)\n{'='*70}")
!python src/train_classifier.py --epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LEARNING_RATE} --protocol all --seed {SEED}


## 6. Aggregate Results Summary Table


In [ ]:
import json
from pathlib import Path
import pandas as pd

lots = ["PUCPR", "UFPR04", "UFPR05"]
rows = []

for fold, lot in enumerate(lots):
    metrics_file = Path(f"outputs/loo_fold{fold}_{lot}_metrics.json")
    if metrics_file.exists():
        with open(metrics_file) as f:
            m = json.load(f)
        rows.append({
            "Protocol": f"LOO Fold {fold}",
            "Evaluation Lot": lot,
            "Accuracy": f"{m.get('accuracy', 0)*100:.2f}%",
            "Precision": f"{m.get('precision', 0):.4f}",
            "Recall": f"{m.get('recall', 0):.4f}",
            "F1 Score": f"{m.get('f1_score', 0):.4f}",
            "ECE (Uncal)": f"{m.get('ece_uncalibrated', 0):.4f}",
            "ECE (Cal)": f"{m.get('ece_calibrated', 0):.4f}",
            "Latency (ms)": f"{m.get('inference_latency_cpu_ms_per_crop', 0):.2f}"
        })

final_file = Path("outputs/all_lots_final_metrics.json")
if final_file.exists():
    with open(final_file) as f:
        m = json.load(f)
    rows.append({
        "Protocol": "Final Model (All Lots)",
        "Evaluation Lot": "Val Split (PUCPR+UFPR04+UFPR05)",
        "Accuracy": f"{m.get('accuracy', 0)*100:.2f}%",
        "Precision": f"{m.get('precision', 0):.4f}",
        "Recall": f"{m.get('recall', 0):.4f}",
        "F1 Score": f"{m.get('f1_score', 0):.4f}",
        "ECE (Uncal)": f"{m.get('ece_uncalibrated', 0):.4f}",
        "ECE (Cal)": f"{m.get('ece_calibrated', 0):.4f}",
        "Latency (ms)": f"{m.get('inference_latency_cpu_ms_per_crop', 0):.2f}"
    })

df_results = pd.DataFrame(rows)
print("\n=== VISION MODULE EVALUATION SUMMARY ===")
print(df_results.to_markdown(index=False))


## 7. Evaluate YOLO11n Slot Mapping Baseline


In [ ]:
!python src/eval_yolo_lots.py


## 8. Backup All Artifacts to Google Drive


In [ ]:
import shutil
drive_save_dir = "/content/drive/MyDrive/smart_parking_vision_largerdata_outputs"
os.makedirs(drive_save_dir, exist_ok=True)

for f in Path("outputs").glob("*"):
    if f.is_file():
        dest = os.path.join(drive_save_dir, f.name)
        shutil.copy(str(f), dest)
        print(f"Backed up: {f.name} -> {drive_save_dir}")
print("\n[SUCCESS] All weights, metrics, diagrams, and repro_check.json backed up to Google Drive!")
